In [156]:
import plotly.express as px
import pandas as pd
import numpy as np

In [ ]:
RESULTS_FILES = [
    "../results/sc-runs/runs-synth-1789418869.csv",
    "../results/sc-runs/runs-ev-1789672175.csv",
]

SCENARIOS = [
    "uniform_everywhere",
    "uniform_randomrows",
    "uniform_lowestrowsumsonly",
    "uniform_halflowestscalars",
]

LOWSC = SCENARIOS[1:]
DISTLABELS = {
    "nandist_euclidean": "nandist Euclidean",
    "dixon_pds_sqeuclidean": "PDS",
    "sqeuclidean": "CC sqeuclidean",
    "eirola_esd_gmm": "ESD/GMM",
    "eirola_esd_mvn": "ESD/MVN",
    "mesquita_eed": "EED",
}
COLORS = px.colors.qualitative.Plotly
DISTCOLORMAP = {
    "nandist_euclidean": COLORS[0],
    "dixon_pds_sqeuclidean": COLORS[1],
    "sqeuclidean": COLORS[2],
    "eirola_esd_gmm": COLORS[3],
    "eirola_esd_mvn": COLORS[4],
    "mesquita_eed": COLORS[5],
}


SCENARIO_LABELS = {
    "uniform_everywhere": "(1) All rows",
    "uniform_randomrows": "(2) 30% of rows, randomly selected",
    "uniform_lowestrowsumsonly": "(3) 30% of rows with lowest sum",
    "uniform_halflowestscalars": "(4) Lower half of z-values",
}

In [158]:
RESULTS_FILE = RESULTS_FILES[0]
DROP_ESDGMM = False

In [159]:
df = pd.read_csv(RESULTS_FILE)
df

,Unnamed: 0,Distance,Run,Missingness,Replicate,CCC,Rand,aRand
0,0,mesquita_eed,uniform_halflowestscalars,70,49,0.812945,1.00,1.000000
1,1,mesquita_eed,uniform_halflowestscalars,70,48,0.814505,1.00,1.000000
2,2,mesquita_eed,uniform_halflowestscalars,70,47,0.799725,1.00,1.000000
3,3,mesquita_eed,uniform_halflowestscalars,70,46,0.729506,0.98,0.959996
4,4,mesquita_eed,uniform_halflowestscalars,70,45,0.826094,1.00,1.000000
...,...,...,...,...,...,...,...,...
19219,19219,sqeuclidean,uniform_everywhere,1,3,0.841503,1.00,1.000000
19220,19220,sqeuclidean,uniform_everywhere,1,2,0.842824,1.00,1.000000
19221,19221,sqeuclidean,uniform_everywhere,1,1,0.847377,1.00,1.000000
19222,19222,sqeuclidean,uniform_everywhere,1,0,0.845208,1.00,1.000000


In [160]:
missigness_vals = sorted(list(set(df["Missingness"])))
distances = sorted(list(set(df["Distance"])))

if DROP_ESDGMM:
    distances.remove("eirola_esd_gmm")
    DISTLABELS["eirola_esd_mvn"] = "ESD"


In [161]:
vals = []

for r in SCENARIOS:
    rfiltered_df = df[df["Run"] == r]
    for m in missigness_vals:
        mfiltered_df = rfiltered_df[rfiltered_df["Missingness"] == m]
        for d in distances:
            dfiltered_df = mfiltered_df[mfiltered_df["Distance"] == d][["CCC", "aRand"]]
            var = dfiltered_df.var(axis=0, numeric_only=True)
            median = dfiltered_df.median(axis=0, numeric_only=True)
            vals.append([r, d, m, median["CCC"], var["CCC"], median["aRand"], var["aRand"]])

condensed_df = pd.DataFrame(
    columns=["Scenario", "Distance", "Missingness in %", "Median CCC", "CCC variance", "Median ARI", "ARI variance"],
    data=vals
)

In [176]:
def quadplot(
    y: str,
    range_y = None,
):
    fig = px.line(
        condensed_df,
        x="Missingness in %",
        y=y,
        color="Distance",
        range_y=range_y,
        facet_col="Scenario",
        facet_col_wrap=2,
        markers=True,
        color_discrete_map=DISTCOLORMAP,
    )
    fig.update_layout(width=1000, height=700)
    fig.for_each_annotation(lambda a: a.update(text=SCENARIO_LABELS[a.text.split("=")[-1]]))
    fig.for_each_trace(lambda a: a.update(name=DISTLABELS[a.name]))
    fig.show()

def threeplot(
    y: str,
    range_y = None,
):
    # Median CCC, highlighted scenarios
    fig = px.line(
        condensed_df[condensed_df["Scenario"].isin(LOWSC)],
        x="Missingness in %",
        y=y,
        color="Distance",
        facet_row="Scenario",
        markers=True,
        color_discrete_map=DISTCOLORMAP,
    )
    fig.update_layout(**THREEPLOT_SIZE)
    fig.update_layout(width=750, height=750)
    fig.for_each_annotation(lambda a: a.update(text=SCENARIO_LABELS[a.text.split("=")[-1]]))
    fig.for_each_trace(lambda a: a.update(name=DISTLABELS[a.name]))
    fig.show()

## CCC

In [177]:
quadplot("Median CCC")
threeplot("Median CCC")

In [178]:
quadplot("CCC variance")
threeplot("CCC variance")

In [166]:
np.nanmax(condensed_df["CCC variance"].to_numpy())

np.float64(0.06166381545646036)

## ARI

In [180]:
quadplot("Median ARI")
quadplot("ARI variance")